In [1]:
# Cell 1 - Import libraries and load raw dataset
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load a sample of 20,000 rows from the raw dataset
# Full dataset is 1 million rows - sampling is standard practice for analysis
df = pd.read_csv('../data/raw/nibss_fraud_dataset.csv', nrows=20000)

print("Dataset loaded successfully")
print(f"Shape: {df.shape}")
print(f"\nColumn names:\n{df.columns.tolist()}")

Dataset loaded successfully
Shape: (20000, 38)

Column names:
['transaction_id', 'customer_id', 'timestamp', 'amount', 'channel', 'merchant_category', 'bank', 'location', 'age_group', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_peak_hour', 'tx_count_24h', 'amount_sum_24h', 'amount_mean_7d', 'amount_std_7d', 'tx_count_total', 'amount_mean_total', 'amount_std_total', 'channel_diversity', 'location_diversity', 'amount_vs_mean_ratio', 'online_channel_ratio', 'is_fraud', 'fraud_technique', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'month_sin', 'month_cos', 'amount_log', 'amount_rounded', 'velocity_score', 'merchant_risk_score', 'composite_risk']


In [2]:
# Cell 2 - Initial data exploration
print("=== BASIC DATA INFO ===")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values per column:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")

=== BASIC DATA INFO ===
Total rows: 20000
Total columns: 38

Data types:
transaction_id           object
customer_id              object
timestamp                object
amount                  float64
channel                  object
merchant_category        object
bank                     object
location                 object
age_group                object
hour                      int64
day_of_week               int64
month                     int64
is_weekend                 bool
is_peak_hour               bool
tx_count_24h            float64
amount_sum_24h          float64
amount_mean_7d          float64
amount_std_7d           float64
tx_count_total            int64
amount_mean_total       float64
amount_std_total        float64
channel_diversity         int64
location_diversity        int64
amount_vs_mean_ratio    float64
online_channel_ratio    float64
is_fraud                  int64
fraud_technique          object
hour_sin                float64
hour_cos                float64

In [3]:
# Cell 3 - Clean the dataset
# Fix timestamp to proper datetime format
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Fill missing fraud_technique with 'Legitimate' since no fraud technique means clean transaction
df['fraud_technique'] = df['fraud_technique'].fillna('Legitimate')

# Extract useful date columns from timestamp
df['transaction_date'] = df['timestamp'].dt.date
df['transaction_month'] = df['timestamp'].dt.month
df['transaction_year'] = df['timestamp'].dt.year
df['transaction_hour'] = df['timestamp'].dt.hour

# Confirm cleaning worked
print("=== AFTER CLEANING ===")
print(f"Missing values remaining:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\nTimestamp dtype: {df['timestamp'].dtype}")
print(f"\nFraud technique values:\n{df['fraud_technique'].value_counts()}")

=== AFTER CLEANING ===
Missing values remaining:
Series([], dtype: int64)

Timestamp dtype: datetime64[ns]

Fraud technique values:
fraud_technique
Legitimate            19943
SOCIAL_ENGINEERING       38
ROBBERY                   7
CARD_THEFT                5
PIN_COMPROMISE            4
PHISHING                  2
OTHER                     1
Name: count, dtype: int64


In [4]:
# Cell 4 - Save cleaned dataset
df.to_csv('../data/cleaned/nibss_banking_cleaned.csv', index=False)

print("Cleaned dataset saved successfully")
print(f"Final shape: {df.shape}")
print(f"\nCleaned file location: ../data/cleaned/nibss_banking_cleaned.csv")
print(f"\nSample of cleaned data:")
print(df[['transaction_id', 'customer_id', 'amount', 'channel', 
          'bank', 'location', 'is_fraud', 'fraud_technique']].head(10))

Cleaned dataset saved successfully
Final shape: (20000, 42)

Cleaned file location: ../data/cleaned/nibss_banking_cleaned.csv

Sample of cleaned data:
     transaction_id    customer_id     amount channel       bank location  \
0  TXN_F08A86FFD87C  CUST_0002AED1   32266.83  Mobile   Sterling    Other   
1  TXN_C2D08134EC83  CUST_0002AED1   72530.49     Web        UBA    Other   
2  TXN_B9499111901D  CUST_0002AED1  168152.87  Mobile       Wema    Other   
3  TXN_48DB1D526A3B  CUST_0002AED1   16439.93  Mobile       FCMB    Other   
4  TXN_56DB1E28B758  CUST_0002AED1    9922.68     POS  FirstBank    Other   
5  TXN_8CB46D78CEED  CUST_0002AED1   80685.56     Web     GTBank    Other   
6  TXN_5F6224E4DA4E  CUST_0002AED1   29276.78     Web       Wema    Other   
7  TXN_8375B147ADE5  CUST_0002AED1  135669.83  Mobile   Fidelity    Other   
8  TXN_EC83B575948F  CUST_0002AED1   24629.69  Mobile   Fidelity    Other   
9  TXN_587A7A626F59  CUST_0002AED1  211626.20      IB   Sterling    Other   

 

In [5]:
# Cell 5 - Data quality summary report
print("=== DATA QUALITY SUMMARY ===")
print(f"\nTotal transactions analysed: {len(df):,}")
print(f"Legitimate transactions: {(df['is_fraud'] == 0).sum():,}")
print(f"Fraudulent transactions: {(df['is_fraud'] == 1).sum():,}")
print(f"Fraud rate: {(df['is_fraud'].mean() * 100):.2f}%")

print(f"\nTransaction channels:")
print(df['channel'].value_counts())

print(f"\nTop 5 banks by transaction volume:")
print(df['bank'].value_counts().head())

print(f"\nTransaction amount summary (NGN):")
print(df['amount'].describe().round(2))

=== DATA QUALITY SUMMARY ===

Total transactions analysed: 20,000
Legitimate transactions: 19,943
Fraudulent transactions: 57
Fraud rate: 0.29%

Transaction channels:
channel
Mobile    8905
Web       4120
POS       3588
IB        1957
ECOM       999
ATM        431
Name: count, dtype: int64

Top 5 banks by transaction volume:
bank
Access       2058
FCMB         2027
Fidelity     2022
GTBank       2010
FirstBank    1991
Name: count, dtype: int64

Transaction amount summary (NGN):
count       20000.00
mean       156487.40
std        335072.67
min           348.74
25%         27954.78
50%         66085.55
75%        160809.32
max      10000000.00
Name: amount, dtype: float64
